In [0]:
#Install extra lib(s)
!pip install -q xlrd
!pip install -q kaggle
!pip install -q kora

import kora
import pandas as pd

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import *

In [0]:
from __future__ import print_function

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
#from sklearn.datasets.samples_generator import make_blobs
from sklearn.datasets import make_blobs
from pyspark import SparkContext
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SQLContext

%matplotlib inline

#Getting Mock Dataset

We also want to have some data to work with. A simple way to get that is to generate it using scikit-learn's method to generate ten blobs in a three-dimensional space.

To make it more realistic we also add an 'id' column of strings, which in real life is often the customer id or the IP address of the IoT device, or similar.

Last we write the dataset as a CSV file, which despite being an awful format, is the one I encounter every day.

In [0]:
n_samples=10000
n_features=3
X, y = make_blobs(n_samples=n_samples, centers=10, n_features=n_features, random_state=42)

# add a row index as a string
pddf = pd.DataFrame(X, columns=['x', 'y', 'z'])
pddf['id'] = 'row'+pddf.index.astype(str)

#move it first (left)
cols = list(pddf)
cols.insert(0, cols.pop(cols.index('id')))
pddf = pddf.loc[:, cols]
pddf.head()

file_path = 'dbfs:/temp/input.csv'

# save the ndarray as a csv file
pddf.to_csv("input.csv", index=False)

In [0]:
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

#threedee = plt.figure(figsize=(12,10)).gca(projection='3d')
fig = plt.figure()
threedee = fig.add_subplot(111, projection='3d')
threedee.scatter(X[:,0], X[:,1], X[:,2], c=y)
threedee.set_xlabel('x')
threedee.set_ylabel('y')
threedee.set_zlabel('z')
plt.show()

#Exploring Data

In [0]:
FEATURES_COL = ['x', 'y', 'z']
path = 'input.csv'

In [0]:
#df = spark.read.options(header="true",inferschema = "true").csv(path)
df = spark.createDataFrame(pddf) 
df.show()

##Check Schema and DataType

In [0]:
df.printSchema()

##Check "Null"

In [0]:
from pyspark.sql.functions import isnan, when, count, col

df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df.describe().show()

In [0]:
##In case you need to remove null
df = df.na.drop()
df.show()

#Prep data for model
Doing some vertorizing

In [0]:
vecAssembler = VectorAssembler(inputCols=FEATURES_COL, outputCol="features")
df_kmeans = vecAssembler.transform(df).select('id', 'features')
df_kmeans.show()

##Optimize choice of k
One disadvantage of KMeans compared to more advanced clustering algorithms is that the algorithm must be told how many clusters, k, it should try to find. 

###Elbow Method
To optimize k we cluster a fraction of the data for different choices of k and look for an "elbow" in the cost function.

(However, may not good due to require to convert to PD)

In [0]:
cost = np.zeros(20)
for k in range(2,20):
    kmeans = KMeans().setK(k).setSeed(1).setFeaturesCol("features")
    model = kmeans.fit(df_kmeans.sample(False,0.1, seed=42))
    cost[k] = model.summary.trainingCost # requires Spark 2.0 or later

In [0]:
fig, ax = plt.subplots(1,1, figsize =(8,6))
ax.plot(range(2,20),cost[2:20])
ax.set_xlabel('k')
ax.set_ylabel('cost')

###Silhouette
The silhouette value is a measure of how similar an object is to its own cluster (cohesion) compared to other clusters (separation). The silhouette ranges from −1 to +1, where a high value indicates that the object is well matched to its own cluster and poorly matched to neighboring clusters. If most objects have a high value, then the clustering configuration is appropriate. If many points have a low or negative value, then the clustering configuration may have too many or too few clusters.

The silhouette can be calculated with any distance metric, such as the Euclidean distance or the Manhattan distance.

In [0]:
from pyspark.ml.evaluation import *

cost = list()
evaluator = ClusteringEvaluator()
for k in range(2,20):
    bkm = KMeans().setK(k).setSeed(1).setFeaturesCol("features")
    bkm_model = bkm.fit(df_kmeans.sample(False,0.1, seed=42))
    tags_predictions = bkm_model.transform(df_kmeans.sample(False,0.1, seed=42))
    silhouette = evaluator.evaluate(tags_predictions)
    cost.append(silhouette)
    
kIdx = np.argmax(cost)


In [0]:
fig, ax = plt.subplots()
plt.plot(range(2,20), cost, 'b*-')
plt.plot(range(2,20)[kIdx], cost[kIdx], marker='o', markersize=12, 
         markeredgewidth=2, markeredgecolor='r', markerfacecolor='None')
plt.xlim(1, plt.xlim()[1])
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Coefficient')
plt.title('Silhouette Scores for k-means clustering')
# Uncomment the next line
display(fig)

### Choose K

In [0]:
k = 5
kmeans = KMeans().setK(k).setSeed(1).setFeaturesCol("features")
model = kmeans.fit(df_kmeans)
centers = model.clusterCenters()

print("Cluster Centers: ")
for center in centers:
    print(center)

#Train Model

In [0]:
transformed = model.transform(df_kmeans).select('id', 'prediction')
rows = transformed.collect()
print(rows[:3])

In [0]:
df_pred = spark.createDataFrame(rows)
df_pred.show()

In [0]:
df_pred = df_pred.join(df, 'id')
df_pred.show()

#Verify Model

In [0]:
pddf_pred = df_pred.toPandas().set_index('id')
pddf_pred.head()

In [0]:
#threedee = plt.figure(figsize=(12,10)).gca(projection='3d')

import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

#threedee = plt.figure(figsize=(12,10)).gca(projection='3d')
fig = plt.figure()
threedee = fig.add_subplot(111, projection='3d')
threedee.scatter(pddf_pred.x, pddf_pred.y, pddf_pred.z, c=pddf_pred.prediction)
threedee.set_xlabel('x')
threedee.set_ylabel('y')
threedee.set_zlabel('z')
plt.show()

In [0]:
model.save("./KMeansModel")

### Load Model ###
#sameModel = KMeansModel.load(sc, ###)